In [ ]:
# Cài đặt thư viện của Hugging Face
!pip install datasets -q

import numpy as np
import re
import torch
import torch.nn as nn
import torch.optim as optim
from datasets import load_dataset
from itertools import chain
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

# Kiểm tra và sử dụng GPU nếu có
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Thiết bị đang sử dụng: {device}")

In [ ]:
# 1. Tải dữ liệu
dataset_dict = load_dataset('Davlan/conll2003_noMISC', trust_remote_code=True)

train_sentences = dataset_dict['train']['tokens']
train_tags = dataset_dict['train']['ner_tags']
val_sentences = dataset_dict['validation']['tokens']
val_tags = dataset_dict['validation']['ner_tags']

# 2. Xây dựng Vocabulary (Từ điển)
all_tokens = list(chain.from_iterable(train_sentences))
vocab = sorted(set(all_tokens))

# Quy định: [PAD] index 0, [UNK] index 1
word_to_ix = {"[PAD]": 0, "[UNK]": 1}
for token in vocab:
    if token not in word_to_ix:
        word_to_ix[token] = len(word_to_ix)

# 3. Xây dựng Tag Map (Ánh xạ nhãn)
unique_ners = sorted(set(list(chain.from_iterable(train_tags))))
tag_to_ix = {tag: i for i, tag in enumerate(unique_ners)}
ix_to_tag = {i: tag for tag, i in tag_to_ix.items()}

print(f'Số lượng từ trong từ điển: {len(word_to_ix)}')
print(f'Số lượng nhãn NER: {len(tag_to_ix)} ({unique_ners})')

In [ ]:
class NERDataset(Dataset):
    def __init__(self, sentences, tags, word_to_ix, tag_to_ix):
        self.sentences = sentences
        self.tags = tags
        self.word_to_ix = word_to_ix
        self.tag_to_ix = tag_to_ix

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):
        words = self.sentences[idx]
        tags = self.tags[idx]
        # Chuyển từ thành index, dùng index 1 nếu từ không có trong từ điển
        sentence_indices = torch.tensor([self.word_to_ix.get(w, 1) for w in words], dtype=torch.long)
        tag_indices = torch.tensor([self.tag_to_ix[t] for t in tags], dtype=torch.long)
        return sentence_indices, tag_indices

def collate_fn(batch):
    sentences, tags = zip(*batch)
    # Pad câu bằng 0, pad nhãn bằng -1 để bỏ qua khi tính Loss
    padded_sentences = pad_sequence(sentences, batch_first=True, padding_value=0)
    padded_tags = pad_sequence(tags, batch_first=True, padding_value=-1)
    return padded_sentences, padded_tags

# Khởi tạo DataLoader
train_dataset = NERDataset(train_sentences, train_tags, word_to_ix, tag_to_ix)
val_dataset = NERDataset(val_sentences, val_tags, word_to_ix, tag_to_ix)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)

In [ ]:
class BiLSTMForNER(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True, bidirectional=True)
        # Hidden_dim * 2 vì là LSTM hai chiều
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x):
        embedded = self.embedding(x)
        lstm_out, _ = self.lstm(embedded)
        logits = self.fc(lstm_out)
        return logits

# Khởi tạo mô hình
EMBEDDING_DIM = 128
HIDDEN_DIM = 128
model = BiLSTMForNER(len(word_to_ix), EMBEDDING_DIM, HIDDEN_DIM, len(tag_to_ix)).to(device)

In [ ]:
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss(ignore_index=-1) # Quan trọng: bỏ qua nhãn -1 (padding)

def evaluate(model, dataloader):
    model.eval()
    total = 0
    correct = 0
    with torch.no_grad():
        for sentences, tags in dataloader:
            sentences, tags = sentences.to(device), tags.to(device)
            logits = model(sentences)
            predictions = torch.argmax(logits, dim=-1)

            mask = (tags != -1)
            correct += (predictions[mask] == tags[mask]).sum().item()
            total += mask.sum().item()
    return correct / total if total > 0 else 0

print("Bắt đầu huấn luyện...")
for epoch in range(5):
    model.train()
    total_loss = 0
    for sentences, tags in train_loader:
        sentences, tags = sentences.to(device), tags.to(device)
        optimizer.zero_grad()
        logits = model(sentences)
        loss = criterion(logits.view(-1, logits.size(-1)), tags.view(-1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    val_acc = evaluate(model, val_loader)
    print(f"Epoch {epoch+1} | Loss: {total_loss/len(train_loader):.4f} | Val Acc: {val_acc:.4f}")

In [ ]:
def predict_sentence(sentence):
    model.eval()
    tokens = re.findall(r"\w+|[^\w\s]", sentence)
    ids = torch.tensor([word_to_ix.get(w, 1) for w in tokens]).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(ids)
        preds = torch.argmax(logits, dim=-1).squeeze(0).tolist()

    return list(zip(tokens, [ix_to_tag[p] for p in preds]))

# Ví dụ thử nghiệm
test_text = "VNU University is located in Hanoi"
result = predict_sentence(test_text)
for word, tag in result:
    print(f"{word:12} : {tag}")